In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd 
import seaborn as sns 

from univariate_analysis import histogram_by_class, boxplot_by_class

In [2]:
%load_ext autoreload
%autoreload 2

# `POS_CASH_BALANCE.csv` Analysis

In [10]:
from column_analysis import basic_data_summary, TypeTable, MissingValueTable, NumericalSummaryTable, CategoricalSummaryTable

pos_df = pd.read_csv("home_credit/POS_CASH_balance.csv")


In [11]:

basic_data_summary(pos_df)

# Type table 
print("\n" + "=" * 100)
print("1. VARIABLE TYPES")
print("=" * 100)
type_table = TypeTable()
print(type_table.plot_table(pos_df).to_string())


# Missing table
print("\n" + "=" * 100)
print("2. MISSING VALUE ANALYSIS")
print("=" * 100)
missing_table = MissingValueTable()
print(missing_table.plot_table(pos_df).to_string())

print("\n" + "=" * 100)
print("3a. NUMERICAL SUMMARY")
print("=" * 100)
numerical_table = NumericalSummaryTable()
print(numerical_table.plot_table(pos_df).to_string())

print("\n" + "=" * 100)
print("3b. CATEGORICAL SUMMARY")
print("=" * 100)
cat_table = CategoricalSummaryTable()
print(cat_table.plot_table(pos_df).to_string())

Shape: 10,001,358 rows x 8 columns
Memory: 1060.9 MB
Fully duplicated rows: 0

1. VARIABLE TYPES
                         dtype       inferred_type  n_unique  unique_ratio  example  memory_mb
column                                                                                        
SK_ID_PREV               int64  numeric (discrete)    936325        0.0936  1803195      76.30
SK_ID_CURR               int64  numeric (discrete)    337252        0.0337   182943      76.30
MONTHS_BALANCE           int64  numeric (discrete)        96        0.0000      -31      76.30
CNT_INSTALMENT         float64  numeric (discrete)        73        0.0000     48.0      76.30
CNT_INSTALMENT_FUTURE  float64  numeric (discrete)        79        0.0000     45.0      76.30
NAME_CONTRACT_STATUS       str         categorical         9        0.0000   Active     526.82
SK_DPD                   int64  numeric (discrete)      3400        0.0003        0      76.30
SK_DPD_DEF               int64  numeric (discret

After reading more detail about this dataset in both the columns description and on the Kaggle website, we understand this dataset to combining all loans a previous customer may have had with home_credit.


__Crucially__, we will isolate only the regular cash loans that a customer previously have had with home_credit, to avoid multicollinearity issues with overlapping information from the credit card information. 

In [12]:
credit_df = pd.read_csv("home_credit/credit_card_balance.csv")

# Any SK_ID_PREV appearing in credit_card_balance.csv is a credit card loan -
# drop those rows from pos_df entirely so what remains is cash/POS loans only,
# free of credit-card-specific behaviour.
pos_non_cc = pos_df[~pos_df["SK_ID_PREV"].isin(credit_df["SK_ID_PREV"])]

print(f"pos_df rows before: {len(pos_df):,}")
print(f"pos_df rows after removing credit-card-overlap SK_ID_PREV: {len(pos_non_cc):,}")
print(f"rows removed: {len(pos_df) - len(pos_non_cc):,} "
      f"({100 * (len(pos_df) - len(pos_non_cc)) / len(pos_df):.2f}%)")

pos_non_cc.head()

pos_df rows before: 10,001,358
pos_df rows after removing credit-card-overlap SK_ID_PREV: 10,001,358
rows removed: 0 (0.00%)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


Zero rows removed - `SK_ID_PREV` never overlaps between `POS_CASH_balance.csv`
and `credit_card_balance.csv` at all, even though their numeric ranges look
similar (~1,000,000-2,843,500 in both). `SK_ID_PREV` identifies one specific
previous application/product, not a customer, so a customer's cash loan and
their credit card are two separate previous applications with their own
distinct `SK_ID_PREV`s - `pos_df` is already cash/POS loans only by
construction, and this filter can never remove anything from it.

100,458 of `pos_df`'s 337,252 customers (`SK_ID_CURR`) also appear in
`credit_card_balance.csv` - so if the goal is avoiding multicollinearity with
credit-card information, that overlap has to be handled at the `SK_ID_CURR`
level during feature engineering/merging (e.g. not double-counting the same
underlying credit behaviour once it's aggregated per customer), not by
filtering rows out of `pos_df` here.

So there does not seem to be that much information here. The suggestion is to move on to analysing another dataset.

# `previous_application.csv` Analysis

In [28]:
from column_analysis import print_all_tables

prev_app_df = pd.read_csv("home_credit/previous_application.csv")
credit_df = pd.read_csv("home_credit/credit_card_balance.csv")
instalments_df = pd.read_csv("home_credit/installments_payments.csv")
POS_balance_df = pd.read_csv("home_credit/POS_CASH_balance.csv")


In [4]:
from column_analysis import print_all_tables, basic_data_summary

basic_data_summary(prev_app_df)
print_all_tables(prev_app_df)

Shape: 1,670,214 rows x 37 columns
Memory: 1703.0 MB
Fully duplicated rows: 0

1. TYPE TABLE
                               dtype                inferred_type  n_unique  unique_ratio                   example  memory_mb
column                                                                                                                        
SK_ID_PREV                     int64         identifier (numeric)   1670214        1.0000                   2030495      12.74
SK_ID_CURR                     int64           numeric (discrete)    338857        0.2029                    271877      12.74
NAME_CONTRACT_TYPE               str                  categorical         4        0.0000            Consumer loans      97.68
AMT_ANNUITY                  float64         numeric (continuous)    357959        0.2143                   1730.43      12.74
AMT_APPLICATION              float64         numeric (continuous)     93885        0.0562                   17145.0      12.74
AMT_CREDIT        

In [8]:
from column_analysis import print_all_tables, basic_data_summary

basic_data_summary(credit_df)
print_all_tables(credit_df)

Shape: 3,840,312 rows x 23 columns
Memory: 846.4 MB
Fully duplicated rows: 0

1. TYPE TABLE
                              dtype                inferred_type  n_unique  unique_ratio   example  memory_mb
column                                                                                                       
SK_ID_PREV                    int64           numeric (discrete)    104307        0.0272   2562384       29.3
SK_ID_CURR                    int64           numeric (discrete)    103558        0.0270    378907       29.3
MONTHS_BALANCE                int64           numeric (discrete)        96        0.0000        -6       29.3
AMT_BALANCE                 float64         numeric (continuous)   1347904        0.3510     56.97       29.3
AMT_CREDIT_LIMIT_ACTUAL       int64           numeric (discrete)       181        0.0000    135000       29.3
AMT_DRAWINGS_ATM_CURRENT    float64         numeric (continuous)      2267        0.0006       0.0       29.3
AMT_DRAWINGS_CURRENT        

In [10]:
from column_analysis import print_all_tables, basic_data_summary

basic_data_summary(instalments_df)
print_all_tables(instalments_df)

Shape: 13,605,401 rows x 8 columns
Memory: 830.4 MB
Fully duplicated rows: 0

1. TYPE TABLE
                          dtype         inferred_type  n_unique  unique_ratio  example  memory_mb
column                                                                                           
SK_ID_PREV                int64    numeric (discrete)    997752        0.0733  1054186      103.8
SK_ID_CURR                int64    numeric (discrete)    339587        0.0250   161674      103.8
NUM_INSTALMENT_VERSION  float64    numeric (discrete)        65        0.0000      1.0      103.8
NUM_INSTALMENT_NUMBER     int64    numeric (discrete)       277        0.0000        6      103.8
DAYS_INSTALMENT         float64    numeric (discrete)      2922        0.0002  -1180.0      103.8
DAYS_ENTRY_PAYMENT      float64    numeric (discrete)      3039        0.0002  -1187.0      103.8
AMT_INSTALMENT          float64  numeric (continuous)    902539        0.0663  6948.36      103.8
AMT_PAYMENT             fl

KeyError: "None of ['column'] are in the columns"

In [29]:
from column_analysis import print_all_tables

basic_data_summary(POS_balance_df)
print_all_tables(POS_balance_df)

Shape: 10,001,358 rows x 8 columns
Memory: 1060.9 MB
Fully duplicated rows: 0

1. TYPE TABLE
                         dtype       inferred_type  n_unique  unique_ratio  example  memory_mb
column                                                                                        
SK_ID_PREV               int64  numeric (discrete)    936325        0.0936  1803195      76.30
SK_ID_CURR               int64  numeric (discrete)    337252        0.0337   182943      76.30
MONTHS_BALANCE           int64  numeric (discrete)        96        0.0000      -31      76.30
CNT_INSTALMENT         float64  numeric (discrete)        73        0.0000     48.0      76.30
CNT_INSTALMENT_FUTURE  float64  numeric (discrete)        79        0.0000     45.0      76.30
NAME_CONTRACT_STATUS       str         categorical         9        0.0000   Active     526.82
SK_DPD                   int64  numeric (discrete)      3400        0.0003        0      76.30
SK_DPD_DEF               int64  numeric (discrete)  

The goal now is to perform a more detailed analysis on some column that accounts only for consumer and cash loans. This will be done by only filtering the entries which have ids that are not already in the credit card dataset.

In [ ]:
non_credit_entries = ~(prev_app_df.SK_ID_PREV.isin(credit_df.SK_ID_PREV))
print("Are there less entries in prev_app_df after removing credit_data?", {non_credit_entries.sum() < len(prev_app_df)})
print(prev_app_df[non_credit_entries]["NAME_CONTRACT_TYPE"].value_counts().to_string())

Are there less entries in prev_app_df after removing credit_data? {np.True_}
NAME_CONTRACT_TYPE
Cash loans         747553
Consumer loans     729151
Revolving loans    100229
XNA                   346



After analysing, we learn that "XNA" actually stands for the category  "Missing" in financial datasets. To handle this entry therefore and homogeneity with the rest of the dataframe, we will replace "XNA" with the word "Missing". Furthermore noting that XNA only occupies a small part of the `NAME_CONTRACT_TYPE` that we are interested in, it's worth dropping these entries (even if they contain loans that might potentially be defaulting.)

In [15]:
prev_app_clean = prev_app_df[~(prev_app_df["NAME_CONTRACT_TYPE"] == "XNA")] 

We now want to obtain information about the previous payment history for the cash loans that we have filtered out. We will do that by filtering out the rows in `installments_payments.csv` that have overlapping ids with `previous_application.csv`. 

Furthermore, we would also like to restrict our attention to loans that have been paid off in the last 12 months only. This will be done in the `installments_payments.csv` dataset. 

We would then like to obtain information on the latest amount paid on each loan, particularly on the loans that are currently active. 

1. We need to filter out loans that are currently active. 
2. We need to then search for the latest amount paid on these loans

We would then like to sum up the total of these loans to get a rough idea on how much instalments the consumer is paying on previous credit within the bank. 

In [17]:
# Restrict installments_payments.csv to loans that also appear in
# prev_app_df (previous_application.csv) - excludes the ~4% of installment
# rows whose SK_ID_PREV has no matching previous-application record.
instalments_matched = instalments_df[instalments_df["SK_ID_PREV"].isin(prev_app_df["SK_ID_PREV"])]

print(f"instalments_df rows before: {len(instalments_df):,}")
print(f"instalments_matched rows after: {len(instalments_matched):,}")

# For each loan, the latest installment is the one with the largest
# (least negative) DAYS_INSTALMENT - the due date closest to the
# application date.
latest_instalment_idx = instalments_matched.groupby("SK_ID_PREV")["DAYS_INSTALMENT"].idxmax()

latest_instalments = instalments_matched.loc[
    latest_instalment_idx, ["SK_ID_PREV", "SK_ID_CURR", "AMT_PAYMENT", "AMT_INSTALMENT", "DAYS_INSTALMENT"]
].rename(columns={"AMT_PAYMENT": "LATEST_AMT_PAID", "AMT_INSTALMENT": "LATEST_AMT_INSTALMENT"})

# DAYS_INSTALMENT is negative (days before application) - flip sign so
# "days ago" reads as a positive distance into the past.
latest_instalments["DAYS_AGO_DUE"] = latest_instalments["DAYS_INSTALMENT"]
latest_instalments = latest_instalments.drop(columns=["DAYS_INSTALMENT"]).set_index("SK_ID_PREV")

latest_instalments.describe()

instalments_df rows before: 13,605,401
instalments_matched rows after: 12,354,575


,SK_ID_CURR,LATEST_AMT_PAID,LATEST_AMT_INSTALMENT,DAYS_AGO_DUE
count,958905.000000,9.577600e+05,9.589050e+05,958905.000000
mean,278306.493054,6.105225e+04,6.123719e+04,-802.945214
std,102782.362219,1.568267e+05,1.566219e+05,778.077959
min,100001.000000,0.000000e+00,0.000000e+00,-2889.000000
25%,189363.000000,6.838864e+03,7.222140e+03,-1331.000000
50%,278489.000000,1.440000e+04,1.470857e+04,-555.000000
75%,367376.000000,3.794016e+04,3.796636e+04,-104.000000
max,456255.000000,3.771488e+06,3.771488e+06,-1.000000


In [27]:
instalments_matched[(instalments_matched["DAYS_ENTRY_PAYMENT"] > -365) & (instalments_matched["AMT_PAYMENT"] == 0)]

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
3945254,2198450,242609,4.0,13,-291.0,-291.0,0.0,0.0
4021596,2028864,289626,2.0,3,-29.0,-28.0,0.0,0.0


In [ ]:
# Loans active in the current month: MONTHS_BALANCE == -1 (the most recent
# observed month, relative to application - POS_CASH_balance.csv never has
# a positive MONTHS_BALANCE) and NAME_CONTRACT_STATUS == "Active".
pos_active_current = POS_balance_df[
    (POS_balance_df["MONTHS_BALANCE"] == -1) & (POS_balance_df["NAME_CONTRACT_STATUS"] == "Active")
]

print(f"POS_balance_df rows: {len(POS_balance_df):,}")
print(f"pos_active_current rows (loans): {len(pos_active_current):,}")

# Same latest_instalments construction as above, restricted to these
# currently-active loans instead of every loan in prev_app_df.
instalments_active_current = instalments_matched[
    instalments_matched["SK_ID_PREV"].isin(pos_active_current["SK_ID_PREV"])
]

latest_instalment_idx_active = instalments_active_current.groupby("SK_ID_PREV")["DAYS_INSTALMENT"].idxmax()

latest_instalments_active = instalments_active_current.loc[
    latest_instalment_idx_active, ["SK_ID_PREV", "SK_ID_CURR", "AMT_PAYMENT", "AMT_INSTALMENT", "DAYS_INSTALMENT"]
].rename(columns={"AMT_PAYMENT": "LATEST_AMT_PAID", "AMT_INSTALMENT": "LATEST_AMT_INSTALMENT"})
latest_instalments_active["DAYS_AGO_DUE"] = latest_instalments_active["DAYS_INSTALMENT"]
latest_instalments_active = latest_instalments_active.drop(columns=["DAYS_INSTALMENT"]).set_index("SK_ID_PREV")

print(f"latest_instalments_active rows (loans matched to an installment record): {len(latest_instalments_active):,}")
latest_instalments_active.describe()

In [ ]:
# Per customer, sum the latest installment amount paid across all of their
# currently-active loans - a rough measure of how much a customer is
# currently paying toward previous credit within the bank.
customer_active_instalment_paid = (
    latest_instalments_active.groupby("SK_ID_CURR")["LATEST_AMT_PAID"].sum()
    .rename("SUM_LATEST_AMT_PAID_ACTIVE")
)

print(f"customers with at least one currently-active loan: {len(customer_active_instalment_paid):,}")
customer_active_instalment_paid.describe()

**Known wrong assumption in `SUM_LATEST_AMT_PAID_ACTIVE`**: for a customer with
more than one currently-active loan, this sums each loan's own most recent
`LATEST_AMT_PAID` - but each loan's "most recent" installment is found
independently, so for a multi-loan customer these payments are **not**
guaranteed to fall in the same month. Summing them together therefore isn't a
"total paid this month" figure; it's closer to "sum of whichever payment
happened to be each loan's latest one," which can span very different dates.

Verified against the data: of 10,691 customers with more than one
currently-active loan in this table, 10,414 (97.4%) have loans whose latest
`DAYS_AGO_DUE` values differ from each other - median gap ~10 days, max gap
2,555 days. So this is a real, common case here, not a rare edge case.

This is a known-wrong simplification, kept deliberately for now to get a
first version of the feature working quickly - revisit by aligning to a
shared month (e.g. summing each loan's payment within the same
`MONTHS_BALANCE` window) before relying on this for anything beyond a rough
first pass.